In [11]:

import sys
from pathlib import Path

ROOT = Path.cwd()
print(ROOT)
if not (ROOT / "src").is_dir():  # handles running from the notebooks/ folder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


/content


In [18]:
import sys
from pathlib import Path
ROOT = Path.cwd()                  # you %cd'd into the repo root, so the root is the current folder
sys.path.append(str(ROOT))
DATA, PRED, RES = ROOT / "data", ROOT / "predictions", ROOT / "results"
PRED.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)

import numpy as np, pandas as pd, torch
from src.data import load_splits
from src.metrics import evaluate, confusion, save_preds
print("GPU:", torch.cuda.is_available(), "| root:", ROOT)

GPU: True | root: /content/ratingProject


# New Section

In [21]:
from src.data import download_labr, load_reviews, load_split_ids, normalize, make_splits, save_splits
if not (DATA / "splits" / "val.csv").exists():
    download_labr(DATA)
    df = load_reviews(DATA); df["norm"] = df.text.map(normalize)
    tr_ids, te_ids = load_split_ids(DATA)
    tr, va, te, log = make_splits(df, tr_ids, te_ids)
    save_splits({"train": tr, "val": va, "test": te}, DATA / "splits")
    print(log)        # should match your PC: train 42681 | val 4743 | test 12651

In [20]:
from transformers import pipeline
zs = pipeline("zero-shot-classification",
              model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
              device=0 if torch.cuda.is_available() else -1)   # 0 = first GPU, -1 = CPU

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [22]:
labels = ["one star", "two stars", "three stars", "four stars", "five stars"]
out = zs(sub.text.iloc[0], candidate_labels=labels,
         hypothesis_template="The reader rated this book {}.")   # {} is replaced by each label -> 5 hypotheses
print("true:", y[0])
print(sub.text.iloc[0][:200])
dict(zip(out["labels"], np.round(out["scores"], 3)))

true: 3
 ممتعة ومشوقة وتشدّ انتباه القارئ . النهاية محبطة !


{'five stars': np.float64(0.243),
 'four stars': np.float64(0.213),
 'two stars': np.float64(0.195),
 'three stars': np.float64(0.189),
 'one star': np.float64(0.159)}

In [27]:
from tqdm.auto import tqdm


def run_zs(texts, labels, template, bs=16):
    P = []
    for out in tqdm(zs(list(texts), candidate_labels=labels, hypothesis_template=template,
                       batch_size=bs), total=len(texts)):
        # the pipeline sorts by score -> we reorder
        s = dict(zip(out["labels"], out["scores"]))
        # back to star order 1..5
        P.append([s[l] for l in labels])
    return np.array(P)


def preds_from(P):
    # column 0 = 1 star
    argmax = P.argmax(1) + 1
    expected = np.clip(np.rint(P @ np.arange(1, 6)), 1,
                       5).astype(int)  # weighted average of stars
    return argmax, expected

In [24]:
variants = {
    "en_stars": (["one star", "two stars", "three stars", "four stars", "five stars"],
                 "The reader rated this book {}."),
    "en_sentiment": (["very negative", "negative", "mixed", "positive", "very positive"],
                     "The opinion of this review is {}."),
    "ar_stars": (["نجمة واحدة", "نجمتين", "ثلاث نجوم", "أربع نجوم", "خمس نجوم"],
                 "قيم القارئ هذا الكتاب ب{}."),
}
results, probs = [], {}
for name, (labs, tpl) in variants.items():
    probs[name] = P = run_zs(sub.text, labs, tpl)
    a, e = preds_from(P)
    results += [evaluate(y, a, f"B_{name}_argmax"),
                evaluate(y, e, f"B_{name}_expected")]
pd.DataFrame(results)

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

,model,macro_f1,accuracy,mae
0,B_en_stars_argmax,0.1981,0.279,1.463
1,B_en_stars_expected,0.1275,0.210,1.207
2,B_en_sentiment_argmax,0.1702,0.256,1.029
3,B_en_sentiment_expected,0.1678,0.266,1.006
4,B_ar_stars_argmax,0.0948,0.100,2.360
5,B_ar_stars_expected,0.1193,0.162,1.527


In [25]:
# A3 predictions for exactly these rows
a3 = pd.read_csv(PRED / "A3_tfidf_LR.csv").set_index("row").loc[sub.row]
results.append(evaluate(a3.y_true, a3.y_pred, "A3_tfidf_LR (same 1000)"))
res = pd.DataFrame(results)
res.to_csv(RES / "phaseB_val1000.csv", index=False)
res

,model,macro_f1,accuracy,mae
0,B_en_stars_argmax,0.1981,0.279,1.463
1,B_en_stars_expected,0.1275,0.210,1.207
2,B_en_sentiment_argmax,0.1702,0.256,1.029
3,B_en_sentiment_expected,0.1678,0.266,1.006
4,B_ar_stars_argmax,0.0948,0.100,2.360
5,B_ar_stars_expected,0.1193,0.162,1.527
6,A3_tfidf_LR (same 1000),0.4287,0.465,0.707


In [26]:
best = res.iloc[:-1].sort_values("macro_f1").iloc[-1].model   # best B row
print(best)
name = best.replace("B_", "").rsplit("_", 1)[0]
a, e = preds_from(probs[name])
p = a if best.endswith("argmax") else e
save_preds(sub, p, "B_zeroshot_best", PRED)
confusion(y, p)

B_en_stars_argmax


,pred 1,pred 2,pred 3,pred 4,pred 5
true 1,11,27,2,0,7
true 2,13,53,3,2,13
true 3,41,85,11,5,54
true 4,50,95,5,9,142
true 5,73,86,10,8,195
